<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-I/blob/main/Clase%20001/ejemplo_1_clase_GridSearchCV_RandomizedSearchCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mejora de modelos de ML parte II
### Coderhouse - Data Science
Profe Jorge Ruiz

Estaremos trabajando en el dataset: **Breast Cancer Wisconsin**

In [1]:
#Importamos librerias y el Dataset
import pandas as pd
import numpy as np
import scipy as sp

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

In [2]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
data

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
         1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
         8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
         8.758e-02],
        ...,
        [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
         7.820e-02],
        [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
         1.240e-01],
        [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
         7.039e-02]]),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0

In [3]:
#Convertimos en dataframe
df = pd.DataFrame(np.c_[data['data'], data['target']],
                  columns= np.append(data['feature_names'], ['target']))

In [4]:
df.shape

(569, 31)

In [5]:
#Visualizamos el objeto
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0.0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0.0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0.0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0.0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0.0


In [6]:
#Como son muchos atributos nos vamos a quedar unicamente con algunos de ellos
features= list(df.columns[0:10])
features

['mean radius',
 'mean texture',
 'mean perimeter',
 'mean area',
 'mean smoothness',
 'mean compactness',
 'mean concavity',
 'mean concave points',
 'mean symmetry',
 'mean fractal dimension']

In [7]:
#A lo que ya tenemos le agregamos la variable: target
data = df[features + ['target']]
data.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,0.0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.0


In [8]:
#Separamos en X e y como así también en Train y Test
X = data.drop(['target'],axis=1)
y = data['target']

# Dividimos los datos en Train y Test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

In [9]:
round(y.value_counts(normalize=True)*100,1)

,proportion
target,
1.0,62.7
0.0,37.3


In [10]:
train_test_split?

In [11]:
X_train.shape, X_test.shape

((426, 10), (143, 10))

In [12]:
#Creamos nuestro objeto RF
clf =  RandomForestClassifier()

# GridSearch CV

In [13]:
#Definicion de Hyperparámetros
param_grid = {'n_estimators': [100, 200],  # Número de árboles en el bosque
    'max_depth': [2,4,6],  # Profundidad máxima de cada árbol
    #'min_samples_split': [2,4,6],  # Número mínimo de muestras requeridas para dividir un nodo interno
    #'min_samples_leaf': [2, 4, 6],  # Número mínimo de muestras requeridas en cada hoja del árbol
    #'bootstrap': [True, False]  # Indica si se debe realizar bootstrap para construir los árboles
}

#Utilizamos la grilla definida anteriormente...
model_RF = GridSearchCV(clf, cv=5, param_grid=param_grid, scoring='f1', n_jobs=-1, verbose=3)

In [14]:
%%time
#Entrenamos nuestro modelo de RF con la grilla ya definida y CV con tamaño de Fold=5
model_RF.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
CPU times: user 350 ms, sys: 22.7 ms, total: 373 ms
Wall time: 19.5 s


GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'max_depth': [2, 4, 6], 'n_estimators': [100, 200]},
             scoring='f1', verbose=3)

In [15]:
model_RF.get_params()

{'cv': 5,
 'error_score': nan,
 'estimator__bootstrap': True,
 'estimator__ccp_alpha': 0.0,
 'estimator__class_weight': None,
 'estimator__criterion': 'gini',
 'estimator__max_depth': None,
 'estimator__max_features': 'sqrt',
 'estimator__max_leaf_nodes': None,
 'estimator__max_samples': None,
 'estimator__min_impurity_decrease': 0.0,
 'estimator__min_samples_leaf': 1,
 'estimator__min_samples_split': 2,
 'estimator__min_weight_fraction_leaf': 0.0,
 'estimator__monotonic_cst': None,
 'estimator__n_estimators': 100,
 'estimator__n_jobs': None,
 'estimator__oob_score': False,
 'estimator__random_state': None,
 'estimator__verbose': 0,
 'estimator__warm_start': False,
 'estimator': RandomForestClassifier(),
 'n_jobs': -1,
 'param_grid': {'n_estimators': [100, 200], 'max_depth': [2, 4, 6]},
 'pre_dispatch': '2*n_jobs',
 'refit': True,
 'return_train_score': False,
 'scoring': 'f1',
 'verbose': 3}

Entonces ... ¿Cómo sabemos cuales son los mejores hyperparámetros? Para ello, tendremos que analizar las siguientes funciones:
 * best_params_
 * best_score_
 * cv_results_

Aclaración: Se recomienda profundizar en la documentación asociada.

Link de Interes:
* https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [16]:
print("Mejores parametros: "+str(model_RF.best_params_))
print("Mejor Score: "+str(model_RF.best_score_)+'\n')

Mejores parametros: {'max_depth': 6, 'n_estimators': 100}
Mejor Score: 0.9476448793052666



In [17]:
#Veamos los resultados obtenidos
scores = pd.DataFrame(model_RF.cv_results_)
scores.sort_values("rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
4,0.322665,0.011835,0.019984,0.000549,6,100,"{'max_depth': 6, 'n_estimators': 100}",0.962963,0.950495,0.938053,0.923077,0.963636,0.947645,0.015454,1
3,0.805140,0.225116,0.032867,0.001356,4,200,"{'max_depth': 4, 'n_estimators': 200}",0.962963,0.950495,0.938053,0.923077,0.954128,0.945743,0.013873,2
5,0.605631,0.055050,0.029627,0.005996,6,200,"{'max_depth': 6, 'n_estimators': 200}",0.953271,0.950495,0.938053,0.923077,0.954128,0.943805,0.011863,3
0,1.066397,0.193794,0.073775,0.029691,2,100,"{'max_depth': 2, 'n_estimators': 100}",0.962963,0.951456,0.921739,0.923077,0.954128,0.942673,0.016984,4
2,0.803982,0.340564,0.041582,0.011153,4,100,"{'max_depth': 4, 'n_estimators': 100}",0.954128,0.941176,0.938053,0.923077,0.954128,0.942113,0.011563,5
1,1.230068,0.287023,0.064409,0.030876,2,200,"{'max_depth': 2, 'n_estimators': 200}",0.944444,0.932039,0.921739,0.923077,0.935780,0.931416,0.008395,6


In [18]:
#Ahora sí ya estamos en condición de realizar nuestras predicciones
prediction = model_RF.predict(X_test)

In [19]:
#Accuracy
print('Exactitud:', accuracy_score(y_test, prediction))

Exactitud: 0.951048951048951


In [20]:
# Matriz de Confusion
cm = confusion_matrix(y_test,prediction)
print("Matriz de confusión:")
print(cm)

Matriz de confusión:
[[49  4]
 [ 3 87]]


# Halving Grid Search

In [21]:
# se recomienda leer sobre https://es.wikipedia.org/wiki/Algoritmo_divide_y_vencer%C3%A1s

#Definicion de Hiperparámetros


param_grid = {'n_estimators': [400,500,600],  # Número de árboles en el bosque
    'max_depth': [18, 20,22],  # Profundidad máxima de cada árbol
    'min_samples_split': [5, 15],  # Número mínimo de muestras requeridas para dividir un nodo interno
    'min_samples_leaf': [4, 6]  # Número mínimo de muestras requeridas en cada hoja del árbol
    #'bootstrap': [True, False]  # Indica si se debe realizar bootstrap para construir los árboles
}
#Aplicamos la grilla al modelo
#Utilizamos la grilla definida anteriormente...
RF_halving_cv = HalvingGridSearchCV(clf, param_grid=param_grid, factor=3, min_resources=40, cv=5,  scoring='f1', n_jobs=-1, verbose=2)


In [22]:
len(X_train)

426

In [23]:
%%time
#Entrenamos RF con la grilla definida arriba y CV con tamaño de Fold=5
RF_halving_cv.fit(X_train, y_train)

n_iterations: 3
n_required_iterations: 4
n_possible_iterations: 3
min_resources_: 40
max_resources_: 426
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 36
n_resources: 40
Fitting 5 folds for each of 36 candidates, totalling 180 fits
----------
iter: 1
n_candidates: 12
n_resources: 120
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 2
n_candidates: 4
n_resources: 360
Fitting 5 folds for each of 4 candidates, totalling 20 fits
CPU times: user 2.02 s, sys: 234 ms, total: 2.26 s
Wall time: 3min 29s


HalvingGridSearchCV(estimator=RandomForestClassifier(), min_resources=40,
                    n_jobs=-1,
                    param_grid={'max_depth': [18, 20, 22],
                                'min_samples_leaf': [4, 6],
                                'min_samples_split': [5, 15],
                                'n_estimators': [400, 500, 600]},
                    scoring='f1', verbose=2)

In [24]:
print("Mejores parametros", RF_halving_cv.best_params_)
print("Mejor CV score", RF_halving_cv.best_score_)
print(f'Accuracy del modelo = {round(accuracy_score(y_test, RF_halving_cv.predict(X_test)), 5)}')

Mejores parametros {'max_depth': 22, 'min_samples_leaf': 4, 'min_samples_split': 15, 'n_estimators': 400}
Mejor CV score 0.9391762290313015
Accuracy del modelo = 0.93706


# Random Search

In [25]:
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html

In [26]:
# Grilla para Random Search
#Definicion de Hyperparámetros
param_grid = {'n_estimators': [100, 200, 300,400,500],  # Número de árboles en el bosque
    'max_depth': [5, 10, 15,20,25],  # Profundidad máxima de cada árbol
    'min_samples_split': [1, 5, 10, 15,20],  # Número mínimo de muestras requeridas para dividir un nodo interno
    'min_samples_leaf': [4, 6, 8, 10],  # Número mínimo de muestras requeridas en cada hoja del árbol
    'bootstrap': [True, False]  # Indica si se debe realizar bootstrap para construir los árboles
}
#Aplicamos la grilla al modelo

model_RF = RandomizedSearchCV(clf, param_grid, cv=5,  scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=2)

In [27]:
%%time
#Entrenamos KNN con la grilla definida arriba y CV con tamaño de Fold=5
model_RF.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
10 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_

CPU times: user 799 ms, sys: 25.5 ms, total: 825 ms
Wall time: 20.6 s


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [5, 10, 15, 20, 25],
                                        'min_samples_leaf': [4, 6, 8, 10],
                                        'min_samples_split': [1, 5, 10, 15, 20],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500]},
                   scoring='neg_root_mean_squared_error', verbose=2)

In [28]:
print("Mejores parametros: "+str(model_RF.best_params_))
print("Mejor Score: "+str(model_RF.best_score_)+'\n')

Mejores parametros: {'n_estimators': 300, 'min_samples_split': 20, 'min_samples_leaf': 6, 'max_depth': 10, 'bootstrap': True}
Mejor Score: -0.25684216006036664



In [29]:
#Analizamos qué obtuvimos
scores = pd.DataFrame(model_RF.cv_results_)
scores

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_max_depth,param_bootstrap,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.981448,0.057072,0.049546,0.004474,200,5,6,20,False,"{'n_estimators': 200, 'min_samples_split': 5, ...",-0.215666,-0.265684,-0.286972,-0.342997,-0.216930,-0.265650,0.047559,3
1,0.917459,0.129511,0.051377,0.018924,300,5,10,5,False,"{'n_estimators': 300, 'min_samples_split': 5, ...",-0.186772,-0.265684,-0.306786,-0.342997,-0.242536,-0.268955,0.053621,5
2,0.001725,0.000615,0.000000,0.000000,500,1,8,20,True,"{'n_estimators': 500, 'min_samples_split': 1, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
3,0.604741,0.013158,0.029090,0.001829,200,20,8,5,True,"{'n_estimators': 200, 'min_samples_split': 20,...",-0.241121,-0.242536,-0.325396,-0.306786,-0.242536,-0.271675,0.036743,7
4,0.914956,0.013157,0.040756,0.000442,300,20,6,10,True,"{'n_estimators': 300, 'min_samples_split': 20,...",-0.186772,-0.242536,-0.286972,-0.325396,-0.242536,-0.256842,0.046747,1
5,0.625670,0.007860,0.031530,0.002654,200,10,6,15,True,"{'n_estimators': 200, 'min_samples_split': 10,...",-0.186772,-0.265684,-0.286972,-0.325396,-0.242536,-0.261472,0.046245,2
6,0.001487,0.000435,0.000000,0.000000,300,1,4,20,False,"{'n_estimators': 300, 'min_samples_split': 1, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
7,0.312356,0.011508,0.016176,0.000731,100,20,6,25,True,"{'n_estimators': 100, 'min_samples_split': 20,...",-0.215666,-0.242536,-0.325396,-0.342997,-0.265684,-0.278456,0.048508,8
8,2.035252,0.427979,0.085432,0.022820,500,5,6,20,True,"{'n_estimators': 500, 'min_samples_split': 5, ...",-0.186772,-0.242536,-0.325396,-0.342997,-0.242536,-0.268047,0.057989,4
9,1.191275,0.220811,0.054203,0.001473,400,20,4,5,False,"{'n_estimators': 400, 'min_samples_split': 20,...",-0.241121,-0.286972,-0.286972,-0.325396,-0.216930,-0.271478,0.038169,6


In [30]:
#Prediccion
prediction = model_RF.predict(X_test)

In [31]:
#Accuracy
print('Exactitud:', accuracy_score(y_test, prediction))

Exactitud: 0.9370629370629371


In [32]:
# Matriz de Confusion
cm = confusion_matrix(y_test,prediction)
print("Matriz de confusión:")
print(cm)

Matriz de confusión:
[[48  5]
 [ 4 86]]
